# 거리(Distance)

군집분석은 항만을 예시로들어보면, 두 입항이 얼마나 비슷한가 같은 것들을 숫자로 나타내는 것이 출발점이다.

선박의 특성값(총톤수, 체류시간 등)을 좌표로 보고, 두 선박 사이의 거리를 헤아린다. 거리를 계산할때 이해하기 쉽게 사용할 수 있는 방법이 유클리디안 거리법(Euclidean Distance)이 있고, 거리가 짧을수록 두 선박이 비슷해보인다 라고 이야기한다.

거리계산에는 중요한 성질이 하나 있는데, 모든 변수가 같은 무게로 더해지는 것이 아니라 숫자 차이가 큰 변수가 거의 결정한다.

총톤수는 수천~수십만 차이날 수 있다. 체류시간은 수시간~수십시간 정도밖에 차이가 안난다. 두 변수를 같이 놓고 거리를 측정하고자 할때, 너무 큰 차이가 있는 총톤수때문에 체류시간은 거의 반영이 안되는 문제가 발생할 수 있다.

## 스케일링(Scaling)

키(cm), 몸무게(kg)로 비교한다고 쳤을때, 키를 mm로 바꿔버리면? 거리 계산 결과가 크게 바뀌게된다. 값의 범위를 비슷하게 맞추는 작업을 수행해야하고 이 작업을 스케일링이라고 일컫는다.

스케일링에서 보편적으로 사용되는 방법은 표준화(Standardization)가 있다. 각 값에서 평균 빼고 표준편차로 나누는 방법이다. 표준화된 값은 평균에서 표준편차 몇 개만큼 떨어져 있는가를 의미하고 이렇게 되면 단위가 사라진다.

## 로그변환(Log Transform)

부산항 입항 선박의 총톤수는 수십톤 20만톤 넘는 선박도 있다. 대부분은 중간이나 작은 선박이고 초거대 선박 소수라고 볼 수 있다. 꼬리가 긴 분포가 발생하게된다.

오른쪽으로 치우쳐진 분포(Right-Skewed Distribution) 이런 변수는 표준화만으로 해결하기 어렵다.

> 왜도(Skewness): 데이터 분포의 좌우 비대칭정도를 나타내는 지표

로그 변환(Log Transform)은 값에다가 로그를 취하는 방식이다. 몇 배 차이인가를 거리로 삼는 방식이다.



# KMeans

KMeans는 가장 보편적으로 사용되는 군집분석 알고리즘이다. 데이터를 미리정한 k개의 군집으로 나누는 방식이다.

K는 군집의 갯수, Means는 각 군집의 중심(Centroid)을 군집에 속한 점들의 평균으로 정하겠다라는 것을 의미한다.

## 원리

KMeans는 두 단계를 번갈아가면서 반복하는 방식으로 동작한다.

- 배정(Assignment): 모든 점을 가장 가까운 중심의 군집에 포함
- 갱신(Update): 각 군집에 속한 점들의 평균으로 중심을 옮김

KMeans가 반복할수록 줄여가나가는 값이 있는데, 이 값은 관성(Inertia)이다. 각 점과 자기 군집 중심사이의 거리의 제곱을 모두 더한 값. 관성을 줄여나가면 점들이 자기 중심 주변에 촘촘히 모이게 된다.


In [6]:
import numpy as np

from sklearn.cluster import KMeans

points = np.array([[-1.2, -0.8], [-1.0, -1.1], [-0.9, -0.6],
                   [1.0, 0.9], [1.3, 1.1], [0.8, 1.4]])

km = KMeans(n_clusters=2, n_init=10, random_state=25)
km.fit(points)

print(km.labels_)
print(km.cluster_centers_.round(3).tolist())
print(round(km.inertia_, 3))

[1 1 1 0 0 0]
[[1.033, 1.133], [-1.033, -0.833]]
0.427


kmeans 모델은 k값을 스스로 결정하는게 아니라, k값을 부여한대로 군집화를 수행한다. 여러 k값으로 군집을 만들어본 다음에 숫자 지표와 해석 가능성을 함꼐 본 다음에 적절한 k값을 찾아서 군집 프로필을 만들어야 한다.

## 엘보 방법(Elbow Method)
k를 늘리면 관성은 항상 줄어든다. 극단적으로 k값을 늘리면 관성은 0이된다. 관성이 가장 작은 k를 고르는 것은 의미가 없고, k를 하나 늘렸을때 관성이 얼마나 줄어드는가를 확인해서 결정하는 것이 합리적이다.

관성이 처음에는 크게 줄어들어들다가 어느지점부터는 거의 줄지 않는데, 꺽이는 지점이 팔꿈치 같다고 해서 엘보 방법이라고 일컫는다.


## 실루엣 계수(Silhouette Coefficient)

실루엣 계수는 한 점이 자기 군집과 얼마나 가깝고 다른 군집과 얼마나 먼지를 -1 ~ 1사이로 나타낸 값이다. 자기 군집 안 다른 점들까지의 평균 거리를 a, 가장 가까운 다른 군집의 점들까지의 평균거리를 b로 두었을 때, b가 a보다 훨씬 크면 1에 가까워지고(잘 분리된 상태), 비슷하면 0 근처(경계에 걸친 상태), a가 더 크면 음수(잘못 배정된 상태)가 된다. 즉, 모든 점의 실루엣을 평균한 값으로 군집 전체의 품질을 확인해보는것도 가능하다.



In [8]:
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [11]:
pipe = make_pipeline(StandardScaler(), KMeans(n_clusters=2, n_init=10, random_state=25))

In [15]:
def func1(*args):
    result = 0
    for _ in args:
        result += _
    return result

func1(1,2,3,4,5)

15